# Water Loss Analytics
## 04 — Persiapan Data Machine Learning
**Raden Satrio Hibatull Rasendriyo**

Analisis SQL dan EDA sudah menjelaskan cakupan laporan, tren NRW, serta konsentrasi kehilangan air. Saya menambahkan machine learning untuk menguji apakah informasi audit suatu sistem pada tahun sebelumnya dapat membantu memperkirakan **volume kehilangan air fisik tahun berikutnya**.

Bagian ML menjadi pelengkap analisis data. Hasilnya nanti dibandingkan dengan baseline sederhana dan dibaca bersama kualitas audit. Model belum digunakan untuk menentukan tindakan teknis atau memperkirakan penghematan.

**Tujuan notebook:** membuat pasangan tahun berurutan, menyusun fitur, dan menetapkan pembagian data yang dapat digunakan ulang di notebook modeling. Notebook ini belum melatih model atau menghitung performa pada test set.


### 1. Rumusan masalah dan batas penggunaan
- **Unit sampel:** satu sistem dengan laporan pada tahun `t` dan `t+1`.
- **Input:** informasi audit tahun `t`.
- **Target:** kehilangan fisik tahun `t+1`, dalam acre-feet (AF).
- **Baseline persistence:** prediksi kehilangan fisik tahun `t+1` sama dengan nilai kehilangan fisik tahun `t`.
- **Evaluasi utama berikutnya:** MAE dalam AF. RMSE membantu melihat kesalahan besar. Rasio dan ukuran sistem juga perlu dilihat saat menafsirkan kesalahan.

Sebagai contoh, fitur audit 2022 dipasangkan dengan target audit 2023. Data 2022 tidak dipasangkan dengan 2024 jika laporan 2023 tidak tersedia.

**Keterbatasan waktu ketersediaan data:** snapshot ini berisi laporan audit yang dapat mengalami revisi. Tanggal publikasi versi historis belum diverifikasi. Fitur tahun `t` belum tentu sudah tersedia pada 1 Januari `t+1`, dan saat model dilatih, label tahun sebelumnya juga belum tentu sudah dipublikasikan. Karena itu, pembagian berdasarkan tahun di sini adalah **evaluasi retrospektif berdasarkan periode audit**, bukan backtest dengan tanggal ketersediaan yang sudah terverifikasi. Klaim prediksi operasional membutuhkan arsip versi dan waktu publikasi laporan.

Tabel input juga sudah disaring menggunakan aturan cleaning, termasuk status laporan pada snapshot saat ini. Hasil ML nantinya berlaku untuk pasangan laporan yang lolos aturan tersebut. Status dan skor validitas tidak digunakan sebagai fitur prediksi karena makna serta waktu pembaruannya perlu ditinjau lebih lanjut.


### Cara menjalankan
Simpan di `water_loss_analytics/notebooks/` dan jalankan **Run All** pada kernel yang sama. Inputnya `data/processed/water_loss_analysis.csv` dari notebook 02.

Notebook memakai pandas, numpy, dan IPython. Hasil akan disimpan ke folder baru `data/ml/` serta `outputs/tables/`. File dengan nama yang sama diperbarui saat dijalankan ulang. Notebook 01–03 dan CSV sumber tidak diubah.


In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = []
for base in [cwd, *cwd.parents]:
    candidates.extend([base, base / 'water_loss_analytics', base / '02_Projects/water_loss_analytics'])
ROOT = next((p for p in candidates if (p / 'data/processed/water_loss_analysis.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Jalankan notebook 02 dahulu; water_loss_analysis.csv belum ditemukan.')
input_path = ROOT / 'data/processed/water_loss_analysis.csv'
input_hash = hashlib.sha256(input_path.read_bytes()).hexdigest()
df = pd.read_csv(input_path, encoding='utf-8-sig', dtype={
    'organization_id':'string','system_id':'string','system_key':'string'})

# Aturan waktu ditetapkan sebelum memilih model atau melihat hasil test.
TRAIN_TARGET_YEARS = list(range(2017, 2022))
VALIDATION_TARGET_YEARS = [2022, 2023]
TEST_TARGET_YEARS = [2024]
base_features = ['water_supplied_af','real_losses_af','apparent_losses_af',
                 'non_revenue_water_af','service_connections','mains_miles']
required = ['source_row','system_key','supplier_name','analysis_year'] + base_features
missing = sorted(set(required) - set(df.columns))
if missing:
    raise ValueError(f'Kolom yang diperlukan belum tersedia: {missing}')
if df.empty or df[['system_key','analysis_year']].isna().any().any():
    raise ValueError('Data kosong atau kunci sistem-tahun tidak lengkap.')
if df.duplicated(['system_key','analysis_year']).any():
    raise ValueError('Ada laporan ganda untuk sistem dan tahun yang sama.')
for col in base_features + ['analysis_year']:
    df[col] = pd.to_numeric(df[col], errors='raise')
if not np.isfinite(df[base_features + ['analysis_year']].to_numpy(dtype=float)).all():
    raise ValueError('Ada angka kosong/tidak hingga; periksa kembali cleaning.')
if not df['analysis_year'].mod(1).eq(0).all():
    raise ValueError('Tahun harus berupa bilangan bulat.')
df['analysis_year'] = df['analysis_year'].astype(int)
assert df[base_features].ge(0).all().all()
assert df[['water_supplied_af','service_connections','mains_miles']].gt(0).all().all()
print(f'Input: {len(df):,} laporan; {df.system_key.nunique():,} ID gabungan sistem.')


Input: 3,507 laporan; 487 ID gabungan sistem.


### 2. Membuat pasangan tahun berurutan
Saya melakukan join berdasarkan **ID gabungan sistem dan tahun target**, bukan berdasarkan urutan baris. Nama penyedia tidak digunakan sebagai kunci karena bisa berubah atau muncul pada beberapa sistem.

ID gabungan organisasi–sistem merupakan kunci kerja dataset ini, bukan identitas yang sudah diverifikasi secara eksternal. Perubahan pemilik atau ID dapat memutus rangkaian laporan. Saya tidak menggabungkan ID yang berbeda secara otomatis.

Kolom target tahun berikutnya diambil secara terbatas: hanya nilai kehilangan fisik, tahun, dan nomor baris sumber untuk penelusuran. Pasokan, NRW, atau komponen audit lain pada tahun target tidak masuk fitur.


In [2]:
source = df[['source_row','system_key','supplier_name','analysis_year'] + base_features].copy()
source = source.rename(columns={
    'source_row':'feature_source_row','analysis_year':'feature_year',
    **{col: col + '_t' for col in base_features}})
source['target_year'] = source['feature_year'] + 1
future = df[['source_row','system_key','analysis_year','real_losses_af']].rename(columns={
    'source_row':'target_source_row','analysis_year':'target_year',
    'real_losses_af':'target_real_losses_af'})

matched = source.merge(future, on=['system_key','target_year'], how='left',
                       validate='one_to_one', indicator=True)
pairs = matched.loc[matched['_merge'].eq('both')].drop(columns='_merge').copy()
pairs['target_source_row'] = pairs['target_source_row'].astype(int)
pairs = pairs.sort_values(['target_year','system_key']).reset_index(drop=True)

# Ringkasan cakupan, tanpa menampilkan distribusi nilai target test.
pair_coverage = matched.groupby('feature_year', as_index=False).agg(
    source_reports=('system_key','size'),
    consecutive_pairs=('_merge', lambda s: int(s.eq('both').sum())))
pair_coverage['without_next_year'] = pair_coverage['source_reports'] - pair_coverage['consecutive_pairs']
pair_coverage['target_year'] = pair_coverage['feature_year'] + 1
print(f'Pasangan berurutan: {len(pairs):,}')
display(pair_coverage)
display(pairs[['system_key','feature_year','target_year','feature_source_row','target_source_row']].head())


Pasangan berurutan: 2,894


,feature_year,source_reports,consecutive_pairs,without_next_year,target_year
0,2016,367,337,30,2017
1,2017,393,369,24,2018
2,2018,403,382,21,2019
3,2019,401,378,23,2020
4,2020,414,384,30,2021
5,2021,413,370,43,2022
6,2022,390,367,23,2023
7,2023,395,307,88,2024
8,2024,331,0,331,2025


,system_key,feature_year,target_year,feature_source_row,target_source_row
0,1004|CA1910043,2016,2017,1032,1033
1,1028|CA3410003,2016,2017,2707,2708
2,1029|CA1910004,2016,2017,857,858
3,102|CA1510001,2016,2017,603,604
4,1030|CA3610043,2016,2017,3159,3160


**Membaca cakupan:** laporan pada tahun terakhir input memang tidak memiliki target tahun berikutnya di tabel ini. Untuk tahun lain, pasangan bisa tidak terbentuk karena laporan tidak tersedia, tidak lolos cleaning, atau identitas berubah. Tabel ini belum membedakan penyebabnya.

Jumlah pasangan yang berhasil dibuat tidak menunjukkan kualitas prediksi. Kualitasnya baru dapat dinilai setelah dibandingkan dengan baseline pada periode yang disisihkan.


### 3. Menyusun fitur dari tahun sebelumnya
Saya mulai dengan enam variabel audit dan dua rasio sederhana. Semua fitur berasal dari tahun `t`.

| Fitur | Alasan digunakan |
|---|---|
| Pasokan air | Memberi konteks skala volume sistem |
| Kehilangan fisik sebelumnya | Menangkap kondisi sebelumnya dan menjadi baseline |
| Kehilangan semu sebelumnya | Memberi konteks komponen kehilangan air |
| NRW sebelumnya | Merangkum air yang tidak menghasilkan pendapatan |
| Jumlah sambungan | Memberi konteks ukuran layanan |
| Panjang pipa utama | Memberi konteks ukuran jaringan |
| Kehilangan fisik / pasokan | Menggambarkan proporsi kehilangan fisik sebelumnya |
| Kehilangan fisik / sambungan | Menggambarkan volume per sambungan sebelumnya |

Sebagian fitur saling berkaitan karena berasal dari neraca air yang sama. Ini perlu dipertimbangkan saat modeling dan interpretasi. Hubungan tersebut tidak menjadi kebocoran target tahun berikutnya selama semua input benar-benar berasal dari tahun sebelumnya, tetapi waktu ketersediaannya tetap memiliki batas yang dijelaskan di awal.

ID, nama, nomor baris, status, skor validitas, dan tahun tidak dimasukkan dalam daftar fitur model awal. Nomor baris dan tahun tetap disimpan sebagai metadata.


In [3]:
pairs['real_loss_share_pct_t'] = 100 * pairs['real_losses_af_t'] / pairs['water_supplied_af_t']
pairs['real_loss_af_per_connection_t'] = pairs['real_losses_af_t'] / pairs['service_connections_t']
FEATURES = [col + '_t' for col in base_features] + [
    'real_loss_share_pct_t','real_loss_af_per_connection_t']
TARGET = 'target_real_losses_af'
BASELINE = 'real_losses_af_t'
assert TARGET not in FEATURES
assert all(name.endswith('_t') for name in FEATURES)
assert np.isfinite(pairs[FEATURES + [TARGET]].to_numpy(dtype=float)).all()
feature_manifest = pd.DataFrame({
    'feature': FEATURES,
    'source_period': ['feature_year (t)'] * len(FEATURES),
    'unit': ['AF','AF','AF','AF','connections','miles','percent','AF/connection/year']
})
display(feature_manifest)
print('Baseline persistence: prediction = real_losses_af_t')


,feature,source_period,unit
0,water_supplied_af_t,feature_year (t),AF
1,real_losses_af_t,feature_year (t),AF
2,apparent_losses_af_t,feature_year (t),AF
3,non_revenue_water_af_t,feature_year (t),AF
4,service_connections_t,feature_year (t),connections
5,mains_miles_t,feature_year (t),miles
6,real_loss_share_pct_t,feature_year (t),percent
7,real_loss_af_per_connection_t,feature_year (t),AF/connection/year


Baseline persistence: prediction = real_losses_af_t


### 4. Membagi data berdasarkan tahun target
| Bagian | Tahun target | Penggunaan |
|---|---|---|
| Training | 2017–2021 | Melatih model awal |
| Validation | 2022–2023 | Membandingkan baseline dan kandidat model |
| Test | 2024 | Menguji model terpilih sekali setelah keputusan modeling selesai |

Saya tidak melakukan random split. Sistem yang sama boleh muncul di beberapa bagian karena tujuan awalnya memperkirakan tahun berikutnya bagi sistem yang memiliki riwayat. Ini bukan pengujian generalisasi ke seluruh sistem baru. Jumlah sistem yang belum muncul pada training dicatat di bawah.

Pada notebook modeling, preprocessing yang belajar dari data, seperti scaling, harus di-fit pada training saja. Setelah model dipilih menggunakan validation, model dapat dilatih ulang pada training + validation sebelum evaluasi final 2024. Keputusan ini tetap tunduk pada keterbatasan waktu publikasi audit.


In [4]:
pairs['split'] = pd.Series(pd.NA, index=pairs.index, dtype='string')
for label, target_years in [('train',TRAIN_TARGET_YEARS),
                            ('validation',VALIDATION_TARGET_YEARS),
                            ('test',TEST_TARGET_YEARS)]:
    pairs.loc[pairs['target_year'].isin(target_years),'split'] = label
if pairs['split'].isna().any():
    raise ValueError('Ada tahun target di luar pembagian yang ditetapkan. Tinjau cakupan sebelum melanjutkan.')
splits = {name:pairs.loc[pairs['split'].eq(name)].copy()
          for name in ['train','validation','test']}
for name,part in splits.items():
    if part.empty:
        raise ValueError(f'Bagian {name} kosong; modeling belum dapat dilanjutkan.')
train_keys = set(splits['train']['system_key'])
train_val_keys = set(pairs.loc[pairs['split'].isin(['train','validation']),'system_key'])
rows=[]
for name,part in splits.items():
    seen = train_val_keys if name == 'test' else train_keys
    rows.append({'split':name,'samples':len(part),'systems':part['system_key'].nunique(),
                 'first_target_year':int(part['target_year'].min()),
                 'last_target_year':int(part['target_year'].max()),
                 'systems_unseen_in_prior_training':len(set(part['system_key'])-seen)})
split_summary=pd.DataFrame(rows)
year_summary=pairs.groupby(['split','target_year'],as_index=False).agg(
    samples=('system_key','size'),systems=('system_key','nunique'))
display(split_summary)
display(year_summary)
print('Untuk validation, sistem baru dibandingkan dengan train. Untuk test, dibandingkan dengan train + validation.')


,split,samples,systems,first_target_year,last_target_year,systems_unseen_in_prior_training
0,train,1850,447,2017,2021,0
1,validation,737,388,2022,2023,15
2,test,307,307,2024,2024,3


,split,target_year,samples,systems
0,test,2024,307,307
1,train,2017,337,337
2,train,2018,369,369
3,train,2019,382,382
4,train,2020,378,378
5,train,2021,384,384
6,validation,2022,370,370
7,validation,2023,367,367


Untuk validation, sistem baru dibandingkan dengan train. Untuk test, dibandingkan dengan train + validation.


### 5. Pemeriksaan konsistensi sebelum modeling
Saya memeriksa bahwa setiap pasangan berjarak satu tahun, tidak ada sistem-tahun target yang ganda, fitur lengkap, dan tahun target antarbagian tidak tumpang tindih.

Nilai audit tahun 2023 boleh menjadi target pada validation sekaligus fitur untuk memprediksi 2024. Itu merupakan penggunaan riwayat berurutan, dengan asumsi audit 2023 sudah tersedia saat prediksi dibuat. **Target 2024 tidak boleh digunakan untuk memilih fitur, model, atau parameter.**

Notebook ini hanya memeriksa struktur dan kelengkapan test set; belum menghitung metrik prediksi maupun menjelajahi distribusi targetnya.


In [5]:
assert pairs['target_year'].sub(pairs['feature_year']).eq(1).all()
assert not pairs.duplicated(['system_key','target_year']).any()
assert splits['train']['target_year'].max() < splits['validation']['target_year'].min()
assert splits['validation']['target_year'].max() < splits['test']['target_year'].min()
assert len(pairs) == sum(len(part) for part in splits.values())
assert all(part[FEATURES + [TARGET]].notna().all().all() for part in splits.values())
assert all(part[TARGET].ge(0).all() for part in splits.values())
# Cocokkan pasangan ke sumber berdasarkan nomor baris, tanpa hanya mengandalkan urutan.
lookup = df.set_index('source_row')
assert np.allclose(lookup.loc[pairs['feature_source_row'],'real_losses_af'].to_numpy(),
                   pairs[BASELINE].to_numpy())
assert np.allclose(lookup.loc[pairs['target_source_row'],'real_losses_af'].to_numpy(),
                   pairs[TARGET].to_numpy())
assert hashlib.sha256(input_path.read_bytes()).hexdigest() == input_hash
print('Pemeriksaan konsistensi lolos. Input cleaning tidak berubah.')


Pemeriksaan konsistensi lolos. Input cleaning tidak berubah.


### 6. Menyimpan data dan keputusan eksperimen
Tiga file split digunakan untuk modeling berikutnya. File konfigurasi menentukan fitur dan target secara eksplisit agar kolom metadata tidak ikut masuk model secara tidak sengaja.

Kesiapan file belum berarti model akan lebih baik daripada baseline. Pada notebook 05 saya akan membandingkan persistence, Ridge, dan Random Forest menggunakan validation. Jika model tidak memberikan perbaikan yang cukup, hasil tersebut tetap dilaporkan sebagai bagian evaluasi proyek.


In [6]:
ml_dir=ROOT / 'data/ml'
table_dir=ROOT / 'outputs/tables'
ml_dir.mkdir(parents=True,exist_ok=True)
table_dir.mkdir(parents=True,exist_ok=True)
for name,part in splits.items():
    path=ml_dir / f'water_loss_{name}.csv'
    part.to_csv(path,index=False,encoding='utf-8-sig')
    print(f'Tersimpan: {path.relative_to(ROOT)} ({len(part):,} sampel)')
for filename,table in [('04_pair_coverage.csv',pair_coverage),
                       ('04_split_summary.csv',split_summary),
                       ('04_samples_by_target_year.csv',year_summary),
                       ('04_feature_manifest.csv',feature_manifest)]:
    table.to_csv(table_dir/filename,index=False,encoding='utf-8-sig')
config={
    'task':'Retrospective next-year real water loss regression',
    'features':FEATURES,'target':TARGET,'target_unit':'acre-feet',
    'baseline_feature':BASELINE,'entity_key':'system_key',
    'train_target_years':TRAIN_TARGET_YEARS,
    'validation_target_years':VALIDATION_TARGET_YEARS,
    'test_target_years':TEST_TARGET_YEARS,
    'input_file':'data/processed/water_loss_analysis.csv','input_sha256':input_hash,
    'samples':{name:len(part) for name,part in splits.items()},
    'primary_metric':'MAE in acre-feet','secondary_metric':'RMSE in acre-feet',
    'publication_timing_verified':False,
    'limitations':[
        'Current audit snapshot may include revisions unavailable historically.',
        'Status-based selection uses current snapshot; results are conditional on cleaning eligibility.',
        'Same system can appear across time splits; not a test solely of unseen systems.',
        'Organization-system identifiers have not been externally reconciled.',
        'Absolute-volume metrics may be driven by large systems; inspect errors by scale.'
    ]
}
(ml_dir/'ml_config.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
print(f'Selesai: {len(pairs):,} pasangan siap untuk modeling retrospektif.')
print('Test 2024 disisihkan; belum ada evaluasi model pada notebook ini.')


Tersimpan: data\ml\water_loss_train.csv (1,850 sampel)
Tersimpan: data\ml\water_loss_validation.csv (737 sampel)
Tersimpan: data\ml\water_loss_test.csv (307 sampel)
Selesai: 2,894 pasangan siap untuk modeling retrospektif.
Test 2024 disisihkan; belum ada evaluasi model pada notebook ini.


### Catatan proyek
Bagian SQL dan EDA tetap menjadi dasar insight tentang cakupan, tren, serta kualitas data. Bagian ML menjawab pertanyaan tambahan: apakah informasi historis memberi prediksi yang lebih baik daripada sekadar memakai nilai sebelumnya?

Dashboard nanti perlu memisahkan **hasil analisis historis** dari **hasil evaluasi prediksi**, dengan menampilkan baseline, periode pengujian, dan keterbatasan audit. Hasil keduanya akan digunakan untuk menyusun cerita proyek pada PPT.
